In [14]:
# 사전 설치 : pip install gradio
from langchain_community.chat_models import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage   # HumanMessage: 사용자가 보낸 메시지, AIMessage : LLM의 메시지
import gradio as gr

In [15]:
# ChatOllama 모델 초기화
model = ChatOllama(model="gemma4:e2b", temperature=0.7, verbose=False)    # temperture가 낮을수록 거의 동일답변, 높을수록 창의적인 답변

In [16]:
# 채팅 기록을 포함하여 응답을 생성하는 함수
def chat(message, history):
    # 이전 대화 기록을 ChatOllama 형식으로 변환
    chat_history = []
    for human, ai in history:
        chat_history.append(HumanMessage(content=human))
        chat_history.append(AIMessage(content=ai))

    # 현재 메시지 추가
    chat_history.append(HumanMessage(content=message))

    # 모델을 사용하여 응답 생성
    response = model.invoke(chat_history)

    return response.content

In [17]:
# Gradio 인터페이스 설정
demo = gr.ChatInterface(
    fn=chat,
    examples=[
        "안녕하세요!",
        "인공지능에 대해 설명해주세요.",
        "파이썬의 장점은 무엇인가요?"
    ],
    title="AI 챗봇",
    description="질문을 입력하면 AI가 답변합니다."
)

In [18]:
# 서버 실행
demo.launch(server_port=7862, server_name="0.0.0.0")

* Running on local URL:  http://0.0.0.0:7862
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "c:\Users\human-31\project\ydataprofiling\.venv\lib\site-packages\gradio\queueing.py", line 856, in process_events
    response = await route_utils.call_process_api(
  File "c:\Users\human-31\project\ydataprofiling\.venv\lib\site-packages\gradio\route_utils.py", line 358, in call_process_api
    output = await app.get_blocks().process_api(
  File "c:\Users\human-31\project\ydataprofiling\.venv\lib\site-packages\gradio\blocks.py", line 2179, in process_api
    result = await self.call_function(
  File "c:\Users\human-31\project\ydataprofiling\.venv\lib\site-packages\gradio\blocks.py", line 1634, in call_function
    prediction = await fn(*processed_input)
  File "c:\Users\human-31\project\ydataprofiling\.venv\lib\site-packages\gradio\utils.py", line 1027, in async_wrapper
    response = await f(*args, **kwargs)
  File "c:\Users\human-31\project\ydataprofiling\.venv\lib\site-packages\gradio\chat_interface.py", line 545, in __wrapper
    return aw

In [47]:
demo.close()

In [2]:
import pandas as pd
from langchain_community.chat_models import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage  # HumanMessage: 사용자가 보낸 메시지, AIMessage : LLM의 메시지
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter   # 특정 문자(예: 줄바꿈, 공백)를 기준으로 텍스트를 분할하는 기능을 제공
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
import gradio as gr

c:\Users\human-31\project\ydataprofiling\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# CSV 파일 로드
df = pd.read_csv("./dataset/indata_kor.csv", encoding='CP949')

In [4]:
df.tail()

,inputs,response
27,한국폴리텍대학 스마트금융과의 최종 아웃풋은 어떤건가요?,스마트금융과는 찍어내기식의 포트폴리오가 아니라 매년 업체에서 요구하는 기술 및 주제...
28,한국폴리텍대학 스마트금융과의 최종 포트폴리오는 어떤건가요?,유튜브 채널에서 스마트금융과를 검색하시면 한국폴리텍대학 스마트금융과 포트폴리오 발표...
29,한국폴리텍대학 스마트금융과 면접시에는 어떤걸 준비하고 가면 될까요?,영문 타자연습 및 스마트금융과에 대한 열정을 보여주면 좋다
30,한국폴리텍대학 스마트금융과 입학 전까지 어떤걸 공부하면 될까요?,기본적인 OA를 잘 다루고 기본코드는 HKCODE의 기본 내용은 보고오면 됨. 파이...
31,한국폴리텍대학 스마트금융과는 대면/비대면 수업 어떻게 진행되나요?,대면으로 진행합니다.


In [5]:
# 텍스트 분할
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)  # 텍스트 조각 최대 1000자, 텍스트 조각 사이에 200자만큼의 중복을 허용(문맥 유지)
texts = text_splitter.split_text("\n".join(df.to_string()))   # 문자열들을 줄바꿈 문자(\n)를 기준으로 연결

In [6]:
# 임베딩 모델 초기화
# embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/distiluse-base-multilingual-cased-v2")
# 모델 이름 : 조직이름(sentence-transformers) 다양한 작업 가능(all)-MS사 경령화 트랜스포머모델(MiniLM)-모델의 레이어수(L6)-모델이 버전(v2)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

C:\Users\human-31\AppData\Local\Temp\ipykernel_9820\3398256464.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2862.61it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
# 벡터 데이터베이스 생성
vectorstore = FAISS.from_texts(texts, embeddings)  # from_texts : 임베딩으로 변환된 벡터를 FAISS 인덱스에 저장

In [8]:
# ChatOllama 모델 초기화
llm = ChatOllama(model="gemma2", tempeature=0.1)  # temperture가 낮을수록 거의 동일답변, 높을수록 창의적인 답변

C:\Users\human-31\AppData\Local\Temp\ipykernel_9820\3185757477.py:2: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  llm = ChatOllama(model="gemma2", tempeature=0.1)  # temperture가 낮을수록 거의 동일답변, 높을수록 창의적인 답변


In [9]:
# 프롬프트 템플릿 정의
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer based on the provided context."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}\n\nContext: {context}")
])

In [10]:
def format_docs(docs):
    if not docs:
        return "No context available"
    result = []
    for doc in docs:
        if hasattr(doc, 'page_content'):
            result.append(doc.page_content)
    return "\n".join(result)

In [11]:
# 리트리버 설정
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

In [12]:
def chat(message, history):
    # 이전 대화 기록을 메시지 형식으로 변환
    chat_history = []
    for human, ai in history:
        chat_history.append(HumanMessage(content=human))
        chat_history.append(AIMessage(content=ai))

    # 문서 검색
    docs = retriever.invoke(message)
    context = format_docs(docs)

    # 프롬프트 생성
    messages = prompt.format_messages(
        chat_history=chat_history,
        question=message,
        context=context
    )

    # 모델 응답 생성
    response = llm.invoke(messages)

    # 소스 문서 정보 추출
    sources = set([doc.metadata.get('source', 'Unknown') for doc in docs])
    source_info = f"\n\n참고 출처: {', '.join(sources)}" if sources else ""

    return response.content + source_info

In [13]:
# Gradio 인터페이스 설정
demo = gr.ChatInterface(
    fn=chat,
    examples=[
        "한국폴리텍대학 스마트금융과 입학 전까지 어떤걸 공부하면 될까요?",
        "스마트금융과에 대해 설명해주세요",
        "한국폴리텍대한 추천할만한 학과 하나를 소개해주세요."
    ],
    title="대학 정보 AI 챗봇",
    description="스마트금융과에 대한 질문을 입력하면 AI가 CSV데이터를 참고하여 한글로 답변합니다."
)

In [16]:
# 서버 실행
demo.launch(server_port=7863, server_name="0.0.0.0")

* Running on local URL:  http://0.0.0.0:7863
* To create a public link, set `share=True` in `launch()`.


In [15]:
demo.close()

Closing server running on port: 7863


In [17]:
import os
from dotenv import load_dotenv  # 환경변수 로드가 필요한 경우
import whisper
import gradio as gr

In [18]:
# .env 파일에서 환경 변수 로드 (필요한 경우)
# load_dotenv()

In [ ]:
# ffmpeg 경로 명시적 설정
# os.environ["FFMPEG_BINARY"] = "C:/aiproject/ffmpeg/bin/ffmpeg.exe"
os.environ["PATH"] += os.pathsep + r"C:\Users\human-31\project\ydataprofiling\ffmpeg\bin"
os.environ["FFMPEG_BINARY"] = r"C:\Users\human-31\project\ydataprofiling\ffmpeg\binffmpeg.exe"